<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 10 · E10 — Cross-schema correctness

Run the grounding ablation on the split fixture. Because the data is byte-identical,
Q1-Q15 keep their answers; the only change is that Q6-Q10/Q12-Q15 are now
cross-schema joins, plus the net-new cross-schema-only Q16-Q18. This measures
whether the multi-schema MDL feature actually enables cross-schema text-to-SQL — and
whether grounding still lifts the score when joins span schemas.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec
import eval_v2 as v2
import seagate_scoring as scoring

FIXTURE = 'seagate_multi'
fdir = v2.fixture_dir(FIXTURE)
manifest = v2.load_table_manifest(FIXTURE)

# Multi-schema project: primary schema seagate_ops + the seagate_core scope.
# seagate_ref is deliberately EXCLUDED (pulling it in is an E9 failure).
config = ec.EvalConfig.from_env(
    schema_name='seagate_ops',
    results_dir=fdir.parent.parent / 'evaluation' / 'results' / 'seagate_multi',
)
client = v2.AgentClientV2(config, schema_names=['seagate_core', 'seagate_ops'])
me = client.login()
print('Authenticated as:', me.get('username') or me)

# Preconditions: Postgres required (R4); memory loop must be off (R1, warn-only).
pre = client.assert_eval_preconditions(require_postgres=True)
print('DB backend:', pre['backend'])
for w in pre['warnings']:
    print('WARNING:', w)

questions = ec.parse_test_queries(fdir / 'test_queries.md')
print('Loaded', len(questions), 'graded questions (Q1-Q18)')
glossary = (fdir / 'bi_glossary.md').read_text(encoding='utf-8')

Authenticated as: admin


DB backend: postgresql
Loaded 18 graded questions (Q1-Q18)


In [2]:
def score_sweep(results):
    """Score a result set with the exact ground-truth scorer (incl. Q16-Q18)."""
    verdicts = {r['id']: scoring.score_result(
        r['id'], r['result_rows'], r['answer_summary']) for r in results}
    correct = sum(1 for v in verdicts.values() if v in scoring.CORRECT_VERDICTS)
    return correct, verdicts

### basic / context_dump (no MDL), then wren_base / wren_bi

In [3]:
print('archived', client.clean_baseline(), 'project(s)')

# basic + context_dump need no MDL.
basic = ec.run_experiment(client, 'basic', questions)
ctx = ec.run_experiment(client, 'context_dump', questions, extra_context=glossary)

# wren_base + wren_bi need the project.
project = client.resolve_project(); pid = project['id']
client.onboard(pid)
wren_base = ec.run_experiment(client, 'wren_base', questions)
doc = client.create_document_from_text(pid, glossary, 'bi_glossary.md')
client.enrich_round(pid, doc['id'], wait_coverage=False)
wren_bi = ec.run_experiment(client, 'wren_bi', questions)

archived 1 project(s)


### Score every condition with the exact scorer (incl. Q16-Q18)

In [4]:
import pandas as pd
conditions = {'basic': basic, 'context_dump': ctx,
              'wren_base': wren_base, 'wren_bi': wren_bi}
rows = []
cross_only = set(scoring.CROSS_SCHEMA_ONLY)
for name, res in conditions.items():
    correct, verdicts = score_sweep(res)
    cross = sum(1 for qid in cross_only if verdicts.get(qid) in scoring.CORRECT_VERDICTS)
    rows.append({'condition': name, 'correct_of_18': correct,
                 'cross_schema_only_of_3': cross})
print(pd.DataFrame(rows).to_string(index=False))

   condition  correct_of_18  cross_schema_only_of_3
       basic              2                       0
context_dump             11                       0
   wren_base              4                       1
     wren_bi             10                       2


### Single-schema vs split delta (load the legacy results if present)

If the legacy `seagate_manufacturing` sweep was scored into `results/*.json`, compare
each condition's single-schema score to the split score above to isolate the
cross-schema penalty (if any). The cross-schema-only column (Q16-Q18) is the direct
pass/fail on the multi-schema feature — it should be 0 without grounding and rise as
the layer learns the cross-schema joins.

In [5]:
# Optional comparison: place legacy per-condition correct counts here once known.
print('Cross-schema-only (Q16-Q18) correctness is the headline E10 number above.')

Cross-schema-only (Q16-Q18) correctness is the headline E10 number above.
